# Validation template. Adapt to your own corpus.

**Reusable participant template.** The pipeline structure follows Day 2 of the workshop. Replace four things and the rest runs unchanged.

1. The CSV path and column names for your texts.
2. The candidate labels for your concept.
3. The thirty hand-coded gold rows for your concept.
4. Your subgroup variable, if any.

Output is Cohen's kappa, accuracy, Pearson r, Spearman rho, a confusion matrix, a subgroup table, and a CSV of per-row scores ready to aggregate.

Runs on free Colab CPU for a few hundred paragraphs. Switch to a paid GPU runtime for thousands or more.

## 0. Setup

In [ ]:
!pip install -q transformers==4.44.2 scikit-learn

In [ ]:
import numpy as np
import pandas as pd
import torch
from transformers import pipeline
from sklearn.metrics import cohen_kappa_score, accuracy_score, confusion_matrix
from scipy.stats import pearsonr, spearmanr

MODEL_ID = 'MoritzLaurer/DeBERTa-v3-base-mnli-fever-anli'
device = 0 if torch.cuda.is_available() else -1
zsc = pipeline('zero-shot-classification', model=MODEL_ID, device=device)
print('Loaded', MODEL_ID, 'on', 'GPU' if device == 0 else 'CPU')

## 1. Plug in your corpus

Replace `CSV_PATH` and `TEXT_COL`. The template assumes one column with the text. Other columns are passed through. If you have a subgroup variable (region, era, party family) point `SUBGROUP_COL` at it. Otherwise leave it as `None`.

In [ ]:
CSV_PATH = '/content/your_corpus.csv'
TEXT_COL = 'text'
SUBGROUP_COL = None  # e.g. 'region' or 'era'. Set to a column name to enable subgroup table.

df = pd.read_csv(CSV_PATH)
df = df.dropna(subset=[TEXT_COL]).reset_index(drop=True)
print('Loaded', len(df), 'rows from', CSV_PATH)

## 2. Define your candidate labels

Three labels are the right starting point. One for each side of your concept and one for the off-topic sink. Wording matters. The model reads each label as a hypothesis and scores entailment, so rewrite for your concept rather than translating mechanically.

In [ ]:
candidate_labels = [
    'the text supports {concept}',
    'the text is critical of {concept}',
    'the text does not discuss {concept}',
]
# Replace {concept} with your concept and re-run.
POSITIVE_LABEL = candidate_labels[0]
NEGATIVE_LABEL = candidate_labels[1]
OFFTOPIC_LABEL = candidate_labels[2]
for lbl in candidate_labels:
    print('-', lbl)

## 3. Build the thirty-row gold set

The single highest-leverage hour of your project. Sample thirty rows from your corpus. Hand-code each with one of the three labels above. The list below is a template. Replace the `(text, label)` pairs with your hand codes.

In [ ]:
GOLD_ROWS = [
    # ('paragraph text from your corpus', 'positive'),
    # ('paragraph text from your corpus', 'negative'),
    # ('paragraph text from your corpus', 'offtopic'),
    # ... thirty rows total ...
]

if not GOLD_ROWS:
    print('Add thirty (text, label) pairs above before continuing.')
else:
    gold = pd.DataFrame(GOLD_ROWS, columns=['text', 'gold_label'])
    print('Gold size', len(gold))
    print(gold['gold_label'].value_counts())

## 4. Score the corpus

One pass over the full corpus. Saves the supportive-class probability and the argmax label. Adjust `batch_size` to your hardware. 16 is safe for free T4 GPU.

In [ ]:
def short_label(top):
    return {POSITIVE_LABEL: 'positive', NEGATIVE_LABEL: 'negative', OFFTOPIC_LABEL: 'offtopic'}[top]

def positive_prob(r):
    return dict(zip(r['labels'], r['scores']))[POSITIVE_LABEL]

results = zsc(df[TEXT_COL].tolist(), candidate_labels=candidate_labels, batch_size=16)
df['top_label'] = [short_label(r['labels'][0]) for r in results]
df['top_score'] = [r['scores'][0] for r in results]
df['p_positive'] = [positive_prob(r) for r in results]
print('Scored', len(df), 'rows.')
print(df['top_label'].value_counts())

## 5. Score the gold set, compute the validation table

Cohen's kappa, accuracy, Pearson r between the supportive probability and a numeric gold score, and Spearman rho. The four numbers your appendix needs.

In [ ]:
if not GOLD_ROWS:
    print('Skipping. Add gold rows in cell 3 first.')
else:
    gold_results = zsc(gold['text'].tolist(), candidate_labels=candidate_labels, batch_size=16)
    gold['pred_label'] = [short_label(r['labels'][0]) for r in gold_results]
    gold['p_positive'] = [positive_prob(r) for r in gold_results]
    label_to_score = {'positive': 1, 'offtopic': 0, 'negative': -1}
    gold['gold_score'] = gold['gold_label'].map(label_to_score)
    gold['pred_score'] = gold['pred_label'].map(label_to_score)

    kappa = cohen_kappa_score(gold['gold_label'], gold['pred_label'])
    acc = accuracy_score(gold['gold_label'], gold['pred_label'])
    r, _ = pearsonr(gold['p_positive'], gold['gold_score'])
    rho, _ = spearmanr(gold['p_positive'], gold['gold_score'])

    print(f'Accuracy        {acc:.2f}')
    print(f"Cohen's kappa   {kappa:.2f}")
    print(f'Pearson r       {r:+.2f}')
    print(f'Spearman rho    {rho:+.2f}')
    print()
    print('Confusion matrix, rows are gold, cols are predicted:')
    cm = confusion_matrix(gold['gold_label'], gold['pred_label'], labels=['positive','negative','offtopic'])
    print(pd.DataFrame(cm, index=['gold positive','gold negative','gold offtopic'], columns=['pred positive','pred negative','pred offtopic']))

## 6. Subgroup table

If your corpus has a subgroup variable, kappa and accuracy by subgroup are what catch comparability problems. Set `SUBGROUP_COL` in cell 1 to enable this.

In [ ]:
if SUBGROUP_COL is None or SUBGROUP_COL not in df.columns:
    print('No subgroup column set. Skipping.')
else:
    subgroup_table = (
        df.groupby(SUBGROUP_COL)
          .agg(n=('top_label', 'size'),
               share_positive=('top_label', lambda s: (s == 'positive').mean()),
               mean_p_positive=('p_positive', 'mean'),
               mean_top_score=('top_score', 'mean'))
          .round(3)
    )
    print(subgroup_table)

## 7. Inspect the disagreements

Off-diagonal rows of the confusion matrix. These are the rows to re-read by hand. Look for patterns. If five disagreements share a phrase you did not anticipate, rewrite a candidate label that covers it. Re-run cell 5.

In [ ]:
if not GOLD_ROWS:
    print('Skipping. Add gold rows in cell 3 first.')
else:
    misses = gold[gold['gold_label'] != gold['pred_label']].copy()
    print(len(misses), 'disagreements')
    for i, r in misses.iterrows():
        print(f"\n[row {i}] gold={r['gold_label']}  pred={r['pred_label']}  p_positive={r['p_positive']:.2f}")
        print('  ' + str(r['text'])[:280])

## 8. Save the per-row CSV

What aggregates to country-year, party-year, or whatever unit your paper uses.

In [ ]:
OUT_PATH = '/content/scored_rows.csv'
df.to_csv(OUT_PATH, index=False)
print('Wrote', len(df), 'rows to', OUT_PATH)

## Iteration loop

If your kappa is below your target, walk through this list before changing models.

1. Re-read 10 disagreements by hand. Group them by failure mode.
2. For each failure mode, rewrite the candidate label that should have caught it. Or add a fourth label that targets it.
3. Re-run cells 4 through 7. Note the new kappa.
4. Stop iterating when kappa stops improving on three rounds in a row.
5. If kappa still will not move, the issue is the codebook, not the model. Re-read your codebook with a co-author.